# Fairlearn lesson: worked solutions to the exercises

Run this notebook top to bottom (about 4 minutes). It uses the same data and model as `fairness.ipynb`, and for
exercise 4 a **temporary MLflow store**, so it changes nothing of yours: your `mlflow.db`, `mlruns/` and registered models are not touched.
Each exercise shows the question, the code, and an **Answer** based on the numbers printed above it.

In [ ]:
import os, sys, shutil, tempfile, warnings
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
warnings.filterwarnings("ignore")
sys.path.insert(0, "../03-mlflow")       # the MLflow lesson's helpers, as in fairness.ipynb
import emr_model as em

import numpy as np
import pandas as pd
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 170, "display.precision", 3, "display.max_columns", 30)
RAW = Path("data/raw")

# Same setup as fairness.ipynb: train on 2024, evaluate on all of 2025
em.simulate("2024-01", 12, out=RAW)
em.simulate("2025-01", 12, out=RAW)
train = em.load_months("2024-01", "2024-12", raw=RAW)
evalset = em.load_months("2025-01", "2025-12", raw=RAW)
model = em.build_model("logistic").fit(train[em.FEATURES], train[em.LABEL])
y = evalset[em.LABEL].to_numpy()
score = model.predict_proba(evalset[em.FEATURES])[:, 1]
flag = (score >= np.quantile(score, 0.8)).astype(int)
print(f"{len(evalset)} visits; baseline sensitivity {flag[y == 1].mean():.3f}, alert rate {flag.mean():.2f}")

## Setup: `equal_sensitivity_flags`, split in two

The notebook's function picks thresholds and applies them in one step. Exercises 2 and 3 need the two steps apart
(choose on one period, apply to another), so here it is as `fit_cuts` + `apply_cuts`. The first cell checks it
gives exactly the notebook's per-sex result.

In [ ]:
def fit_cuts(scores, groups, outcomes, capacity=0.20):
    """Per-group thresholds with equal sensitivity, as high as `capacity` allows (same logic as the notebook)."""
    best = None
    for target in np.arange(0.40, 0.90, 0.005):
        cuts = {}
        for g in np.unique(groups):
            in_g = groups == g
            cases = np.sort(scores[in_g & (outcomes == 1)])[::-1]
            cuts[g] = cases[int(np.ceil(target * len(cases))) - 1]
        flags = apply_cuts(scores, groups, cuts)
        if flags.mean() > capacity:
            break
        best = (cuts, target)
    return best

def apply_cuts(scores, groups, cuts):
    return np.array([s >= cuts[g] for s, g in zip(scores, groups)])

def equal_sensitivity_flags(scores, groups, capacity=0.20):          # the notebook's version, for the check
    best = None
    for target in np.arange(0.40, 0.90, 0.005):
        flags = np.zeros(len(scores), bool)
        for g in np.unique(groups):
            in_g = groups == g
            cases = np.sort(scores[in_g & (y == 1)])[::-1]
            flags[in_g] = scores[in_g] >= cases[int(np.ceil(target * len(cases))) - 1]
        if flags.mean() > capacity:
            break
        best = flags
    return best

sex = evalset["sex"].to_numpy()
cuts_sex, tgt = fit_cuts(score, sex, y)
print("matches the notebook:", (apply_cuts(score, sex, cuts_sex) == equal_sensitivity_flags(score, sex)).all(),
      "| target sensitivity", round(tgt, 3))

def table(flags, groups, outcomes, label="group"):
    d = pd.DataFrame({"g": groups, "y": outcomes, "f": flags})
    t = d.groupby("g").apply(lambda x: pd.Series({"visits": len(x), "GAM cases": x.y.sum(),
                                                  "prevalence": x.y.mean(), "alert_rate": x.f.mean(),
                                                  "sensitivity": x.f[x.y == 1].mean()}))
    t.index.name = label
    return t

## Exercise 1

**Per-region thresholds.** Apply `equal_sensitivity_flags` to `evalset["region"]` instead of sex. What happens to Addis Ababa, to Afar and Somali, and to overall sensitivity? Who pays for equality, and would the nutrition programme accept that?

In [ ]:
region = evalset["region"].to_numpy()
cuts_region, tgt_r = fit_cuts(score, region, y)
flags_region = apply_cuts(score, region, cuts_region)

before = table(flag, region, y, "region")
after = table(flags_region, region, y, "region")
cmp = before[["prevalence", "GAM cases", "alert_rate", "sensitivity"]].join(
      after[["alert_rate", "sensitivity"]], lsuffix=" (national threshold)", rsuffix=" (per-region)")
cmp["flags before"] = (before["alert_rate"] * before["visits"]).round().astype(int)
cmp["flags after"] = (after["alert_rate"] * after["visits"]).round().astype(int)
display(cmp.sort_values("prevalence"))
print(f"overall sensitivity: {flag[y == 1].mean():.3f} -> {flags_region[y == 1].mean():.3f}   "
      f"(target {tgt_r:.3f}); alert rate {flag.mean():.3f} -> {flags_region.mean():.3f}")
print("PPV (share of flags that are real):", round(y[flag == 1].mean(), 3), "->", round(y[flags_region].mean(), 3))

**Answer.** Per-region thresholds make every region's sensitivity almost the same, about **0.556** (table above), and the cost is paid by the highest-need regions.
- **Addis Ababa** gains most: sensitivity 0.154 to 0.558, with flags rising from 43 to 262. (Only 52 cases, so treat it as roughly, not exactly.)
- **Afar and Somali** lose: 0.801 to 0.559 and 0.794 to 0.557. Their flags fall from 443 to 242 and 581 to 320, because the capacity moves elsewhere.
- **Overall sensitivity falls** from 0.577 to 0.556, and the share of flags that are real (PPV) from 0.319 to 0.310: flags in a region where few children
  are malnourished (2.6% in Addis) catch fewer cases than flags in Afar (19.6%).

Equality is paid for by children in Afar and Somali, who are now less likely to be caught, and by the programme overall. I doubt a nutrition programme
would accept that; the better answer is more capacity for Addis or a cheap second screen there, rather than taking flags away from the regions
with the most malnutrition. It is the argument for choosing the fairness definition (and which groups) with the programme owners first.

## Exercise 2

**Thresholds set in advance.** In practice thresholds are chosen on past data. Choose per-sex thresholds on Jan-Jun 2025 and apply them unchanged to Jul-Dec 2025. Does equal sensitivity hold? What about the alert rate (the lean season starts in June)?

In [ ]:
month = evalset["month"].to_numpy()
h1, h2 = month <= "2025-06", month >= "2025-07"

cuts_h1, tgt1 = fit_cuts(score[h1], sex[h1], y[h1])                # choose on Jan-Jun only
print("thresholds chosen on Jan-Jun:", {g: round(float(v), 3) for g, v in cuts_h1.items()}, "| target", round(tgt1, 3))
flags_h1 = apply_cuts(score[h1], sex[h1], cuts_h1)
flags_h2 = apply_cuts(score[h2], sex[h2], cuts_h1)                 # applied unchanged to Jul-Dec

for label, fl, mask in [("Jan-Jun (where chosen)", flags_h1, h1), ("Jul-Dec (new data)", flags_h2, h2)]:
    t = table(fl, sex[mask], y[mask], "sex")
    print(f"{label}: alert rate {fl.mean():.3f}  overall sensitivity {fl[y[mask] == 1].mean():.3f}  "
          f"girls {t.loc['F', 'sensitivity']:.3f}  boys {t.loc['M', 'sensitivity']:.3f}  "
          f"gap {t.loc['M', 'sensitivity'] - t.loc['F', 'sensitivity']:+.3f}")

by_month = pd.DataFrame({"month": month[h2], "flag": flags_h2, "y": y[h2], "sex": sex[h2]})
print("monthly alert rate and prevalence, Jul-Dec:")
by_month.groupby("month").agg(alert_rate=("flag", "mean"), prevalence=("y", "mean")).T

In [ ]:
# Is the Jul-Dec gap bigger than chance? Bootstrap the girls/boys gap on Jul-Dec.
rng = np.random.default_rng(0)
idx_all = np.where(h2)[0]
gaps = []
for _ in range(300):
    i = rng.choice(len(idx_all), len(idx_all))
    yy, ff, ss = y[h2][i], flags_h2[i], sex[h2][i]
    gaps.append(ff[(yy == 1) & (ss == "M")].mean() - ff[(yy == 1) & (ss == "F")].mean())
print("Jul-Dec gap (boys - girls), 90% interval:", np.round(np.quantile(gaps, [0.05, 0.95]), 3))

**Answer.** Equal sensitivity **mostly holds, the alert rate does not.**
- **Sensitivity:** exactly equal on Jan-Jun (0.572 vs 0.571) by construction. On Jul-Dec girls 0.656, boys 0.675, gap +0.019,
  and the 90% interval (-0.018 to 0.053) includes zero, so it is consistent with equal. The baseline gap was about 0.07.
- **Alert rate:** chosen to give 20% on Jan-Jun (0.197), it becomes **0.256** on Jul-Dec. By month: 0.33 to 0.34 in July to September,
  then 0.17 to 0.18 in October to December. The lean season (June to September) raises risk scores (prevalence 15 to 17% against 9%),
  so fixed score cut-offs flag far more than the clinic can follow up. The higher overall sensitivity (0.665) is only that extra
  flagging, which the clinic cannot deliver.
- Fixes: re-fit or re-check the thresholds each month, or fix the *share* flagged (top 20% within each sex each month) instead of a score; and monitor alert rate and sensitivity by group.

## Exercise 3

**Intersectional thresholds.** Use `sex` and `age_band` together (6 groups). With bootstrap intervals (`MetricFrame(..., n_boot=200)`), which of these thresholds would you trust?

In [ ]:
inter = (evalset["sex"].astype(str) + " " + evalset["age_band"].astype(str)).to_numpy()
cuts6, tgt6 = fit_cuts(score, inter, y)
flags6 = apply_cuts(score, inter, cuts6)

# Bootstrap intervals (90%) for the new flags
mf = MetricFrame(metrics={"sensitivity": true_positive_rate}, y_true=y, y_pred=flags6.astype(int),
                 sensitive_features=evalset[["sex", "age_band"]], n_boot=200, ci_quantiles=[0.05, 0.95], random_state=0)
lo, hi = mf.by_group_ci
cases = evalset[evalset[em.LABEL] == 1].groupby(["sex", "age_band"], observed=True).size()
res = pd.DataFrame({"threshold": [cuts6[f"{s} {a}"] for s, a in mf.by_group.index],
                    "GAM cases": cases, "sens (6-group thresholds)": mf.by_group["sensitivity"],
                    "low": lo["sensitivity"], "high": hi["sensitivity"]})
res["interval width"] = res["high"] - res["low"]
res["sens (single threshold)"] = MetricFrame(metrics=true_positive_rate, y_true=y, y_pred=flag,
                                             sensitive_features=evalset[["sex", "age_band"]]).by_group
display(res.round(3))
print(f"overall sensitivity {flag[y == 1].mean():.3f} -> {flags6[y == 1].mean():.3f}; alert rate {flags6.mean():.3f}; target {tgt6:.3f}")

In [ ]:
# Stability check: choose the six thresholds on Jan-Jun, how far do they move when chosen on Jul-Dec instead?
c_a, _ = fit_cuts(score[h1], inter[h1], y[h1]); c_b, _ = fit_cuts(score[h2], inter[h2], y[h2])
pd.DataFrame({"Jan-Jun": c_a, "Jul-Dec": c_b}).assign(difference=lambda d: d["Jul-Dec"] - d["Jan-Jun"]).round(3)

**Answer.** Six thresholds equalise sensitivity at about 0.57 in every group (table above), closing the 6-23 month gap
(girls 0.589 vs boys 0.683 with a single threshold) at the price of overall sensitivity 0.577 to 0.567. But the groups are not equally well estimated:
- **Trust most: girls and boys at 6-23 months.** 754 and 861 GAM cases, the narrowest intervals (width 0.056 and 0.053).
- **Trust least: the 0-5 month groups** (about 200 cases each, interval width 0.12) and, a little better, **24-59 months** (285 to 351 cases, width 0.09 to 0.10).
  Their thresholds rest on a few hundred cases, and an interval of +-0.05 to +-0.06 around "equal" cannot show a smaller difference.
- **Stability check:** thresholds chosen on Jan-Jun against Jul-Dec move by 0.02 to 0.06, all upward (the lean season again), most for boys at 6-23 months. Part of that is the season, not noise,
  which is another reason to re-fit regularly.

Six groups share the same 20% capacity and the same data, so more groups means more estimates that are individually noisier. A sensible compromise: use the age-specific thresholds for 6-23 months and keep one pooled threshold for the thinner groups, until more data arrives.

## Exercise 4

**Log it.** In the MLflow lesson, add `MetricFrame(...).by_group` for sex and region to each run (`mlflow.log_dict`) and the girls/boys sensitivity gap as a metric. Then tighten the promotion rule to require a gap below 0.03.

In [ ]:
# A temporary MLflow store, and the MLflow lesson's train_and_log (data goes to ../03-mlflow/data, which is gitignored)
import mlflow
from mlflow import MlflowClient
from train import EXPERIMENT, train_and_log        # from ../03-mlflow (on sys.path above)

TMP = Path(tempfile.mkdtemp(prefix="fairlearn_solutions_"))
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{TMP / 'mlflow.db'}"
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
exp_id = mlflow.create_experiment(EXPERIMENT, artifact_location=(TMP / "artifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)

em.simulate("2024-01", 12)
em.simulate("2025-01", 6, scenario="concept_shift")

In [ ]:
def log_fairness(run_id, model_uri, test_df):
    """Add per-group MetricFrame tables and the girls/boys gap to a finished run (or put this inside train_and_log)."""
    mod = mlflow.sklearn.load_model(model_uri)
    yt = test_df[em.LABEL].to_numpy()
    sc = mod.predict_proba(test_df[em.FEATURES])[:, 1]
    fl = (sc >= np.quantile(sc, 0.8)).astype(int)
    metrics = {"sensitivity": true_positive_rate, "alert_rate": selection_rate}
    out = {}
    for col in ["sex", "region"]:
        mf = MetricFrame(metrics=metrics, y_true=yt, y_pred=fl, sensitive_features=test_df[col])
        out[col] = mf.by_group.round(4).to_dict(orient="index")            # {group: {metric: value}}
        if col == "sex":
            gap = float(mf.by_group.loc["M", "sensitivity"] - mf.by_group.loc["F", "sensitivity"])
    with mlflow.start_run(run_id=run_id):
        mlflow.log_dict(out, "metricframe_by_group.json")
        mlflow.log_metric("sensitivity_gap_boys_minus_girls", gap)
    return gap

def model_uri_of(run_id):
    return mlflow.search_logged_models(filter_string=f"source_run_id = '{run_id}'", experiment_ids=[exp_id],
                                       output_format="list")[0].model_uri

benchmark = em.load_months("2025-05", "2025-06")
# champion: 2024 model (the best 2024 run in notebook 2 is logistic C=1); challenger: retrained to April 2025
champ_run, _, champ_uri = train_and_log(model="logistic", C=1.0, run_name="champion-2024")
chall_run, _, chall_uri = train_and_log(model="logistic", C=1.0, train_until="2025-04",
                                        test_from="2025-05", test_until="2025-06", run_name="challenger-2025-04")
# For the fair comparison both models are scored on the same May-Jun 2025 benchmark
gaps = {"champion (v1)": log_fairness(champ_run, champ_uri, benchmark),
        "challenger": log_fairness(chall_run, chall_uri, benchmark)}
print({k: round(v, 4) for k, v in gaps.items()})
print(mlflow.artifacts.load_dict(f"runs:/{chall_run}/metricframe_by_group.json")["sex"])

In [ ]:
def should_promote(champ, chall, min_auc_gain=0.005, max_group_drop=0.02, max_gap=0.03):
    reasons = []
    if chall["roc_auc"] - champ["roc_auc"] < min_auc_gain:
        reasons.append(f"ROC-AUC gain {chall['roc_auc'] - champ['roc_auc']:+.4f} < {min_auc_gain}")
    for g in ["sens_girls", "sens_boys"]:
        if champ[g] - chall[g] > max_group_drop:
            reasons.append(f"{g} drops by {champ[g] - chall[g]:.3f}")
    if chall["gap"] >= max_gap:                                       # the new, tighter condition
        reasons.append(f"girls/boys gap {chall['gap']:.3f} is not below {max_gap}")
    return not reasons, reasons

rows = {}
for name, uri in [("champion (v1)", champ_uri), ("challenger", chall_uri)]:
    overall, by_group = em.evaluate(mlflow.sklearn.load_model(uri), benchmark)
    rows[name] = {"roc_auc": overall["roc_auc"], "sens_girls": by_group["sex"]["F"]["sensitivity"],
                  "sens_boys": by_group["sex"]["M"]["sensitivity"]}
    rows[name]["gap"] = rows[name]["sens_boys"] - rows[name]["sens_girls"]
comparison = pd.DataFrame(rows).T
display(comparison.round(4))
print("old rule (no gap condition):", should_promote(comparison.loc["champion (v1)"], comparison.loc["challenger"], max_gap=9))
print("tightened rule (gap < 0.03):", should_promote(comparison.loc["champion (v1)"], comparison.loc["challenger"]))
shutil.rmtree(TMP, ignore_errors=True)       # remove the temporary store

**Answer.** In `train_and_log`, after `em.evaluate`, add a `MetricFrame` for sex and for region, `mlflow.log_dict(tables, "metricframe_by_group.json")`, and
`mlflow.log_metric("sensitivity_gap_boys_minus_girls", gap)`; the cells above do the same on finished runs (the artifact for the challenger is printed).
On the May-Jun 2025 benchmark the gap is **0.035 for the champion and 0.047 for the challenger**. The old rule **promotes** the challenger
(better ROC-AUC, no group worse), but the tightened rule (gap below 0.03) **refuses** it: "girls/boys gap 0.047 is not below 0.03".
Note the champion's own gap (0.035) would not pass either, so as written the rule blocks the current model's successors until the gap is fixed (for example with the
per-sex thresholds of `fairness.ipynb` section 3) or the tolerance is set deliberately. Two months of data is a short benchmark for a gap measure: report it with an interval (`n_boot`).

## Exercise 5

**Resume matching.** Sketch the same analysis for shortlisting candidates: what is the "miss", which groups, which definition, and which mitigations are allowed?

**Answer** (a thinking exercise, no code).

- **The miss:** a qualified candidate who is not shortlisted (a false negative). Capacity is the same kind of
  constraint as the clinic's 20%: a recruiter only reads so many CVs. A false alarm (an unqualified candidate shortlisted) costs a
  recruiter's time, which is cheap compared with the miss, which costs a person a job.
- **The catch:** "qualified" is not observed the way GAM is. The label is usually past hiring decisions, which already contain
  human bias, so the model can learn it. Here the label is the real outcome (malnourished or not).
- **Groups:** sex, age, ethnicity, disability, and their intersections (for example older women), plus proxies
  (name, address, school, career gaps). Small groups need intervals (`n_boot`).
- **Definition:** equal opportunity: among candidates who are qualified, each group has the same chance of being
  shortlisted. Demographic parity is the "four-fifths" adverse-impact rule used in employment law in some countries;
  know which one applies. Decide with HR and legal first, as with the programme owners here.
- **Allowed mitigations:** those that do not use the protected attribute at decision time: remove proxies and
  anonymise CVs, rebalance or reweigh training data, reductions such as `ExponentiatedGradient` (a yes/no decision fits
  here), audit and monitor outcomes by group, and keep a human in the loop. **Not allowed** in most places:
  per-group thresholds, because they treat candidates differently by sex or ethnicity (this was fine in the clinic, where growth
  standards already differ by sex). Check the local law before assuming either way.